In [ ]:
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
# Task 1: Write your code here:
Q1_data_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(Q1_data_path)# YOUR CODE HERE


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt

plt.figure(figsize=[6,6])
plt.hist(df['Delivery_Time'].dropna(), bins=30, edgecolor='Black')
plt.title('Target Distribution')
plt.ylabel('Freq')
plt.xlabel('Delivery_Time')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_no_order_id = df.drop('Order_ID',axis=1)

print(df_no_order_id.head())

In [ ]:
# Task 2: Write your code here:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
import numpy as np


df_no_order_id['Delivery_Time']=df_no_order_id['Delivery_Time'].fillna(np.mean(df['Delivery_Time']))

In [ ]:
# Task 3: Write your code here:
print(df_no_order_id.duplicated().sum())
df_no_order_id = df_no_order_id.drop_duplicates()
print(df_no_order_id.describe())
print(df_no_order_id.info())


In [ ]:
# Task 4: Write your code here:
#4. **Encode categorical variables** if needed (Bonus if used One Hot Encoding)
object_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']

LEncoder = LabelEncoder()
for col in object_cols:
  df_no_order_id[col] = LEncoder.fit_transform(df_no_order_id[col])

In [ ]:
# Task 5: Write your code here:
#5. **Apply feature scaling for all features** (Use StandardScaler)
scaler = StandardScaler()

features = ['Distance_km','Preparation_Time_min','Courier_Experience_yrs'] + object_cols
df_no_order_id[features] = scaler.fit_transform(df_no_order_id[features])

df_clean = df_no_order_id.copy()

print(df_clean.head())
print(df_clean.isnull().sum())
print(df_clean.shape)
df_clean = df_clean.dropna(axis=0)

print(df_clean.isnull().sum())
print(df_clean.shape)

In [ ]:
# Task 6: Write your code here:
#6. **Check for target imbalance and state if it is imbalanced or not** (keep this cell empty if not needed)

In [ ]:
# Task 1: Write your code here:
X = df_clean[features].copy()
y = df_clean['Delivery_Time'].copy()

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
'''
Use the correct split: KFold OR StratifiedKFold
Train a RandomForest model
Evaluate using MAE (Mean Absolute Error) ONLY
Print the averaged score across all folds
'''
kf = KFold(n_splits=5, shuffle=True, random_state=42)

fold_losses = []

model = RandomForestRegressor(n_estimators=100, random_state=42)

for fold, (train_index, val_index) in enumerate(kf.split(X)):
  print(f'Fold {fold + 1}/{5}')
  X_train_fold, X_val_fold = X.iloc[train_index], X.iloc[val_index]
  y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[val_index]

  model.fit(X_train_fold, y_train_fold)
  y_pred = model.predict(X_val_fold)
  mae = mean_absolute_error(y_val_fold, y_pred)

  fold_losses.append(mae)

average_losses = np.mean(fold_losses)
print(f"MAE Average :{average_losses}")




In [ ]:
# Task 1: Write your code here:
# Feature importance
importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 6))
plt.plot(fold_losses, label='Average Loss')
plt.xlabel('Iteration')
plt.ylabel('MAE Loss')
plt.title('Linear Regression Loss (Averaged Across Folds)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Task Bonus: Write your code here: